In [ ]:
#Google Sheets Linking (Needs user authorization)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
import pandas as pd

# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/14pMQzQwy2Dl9OIhS24hyJdX1sfEdVi9SeZfGuYaMKQ0/edit?usp=drive_link'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

print("Original Data:")
display(df.head())

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

def preprocessing(text):
  #collapse spaces to make the text more concise
  text = re.sub(r'\s+', ' ', text)

  #Tokenize links + emails
  text = re.sub(r'http\S+', 'URL_TOKEN', text)
  text = re.sub(r'\S*@\S*\s?', 'EMAIL_TOKEN', text)

  #regex to keep alphabetic characters, both capitalized and lowercase, numbers, spaces, . , ! ? ' \ " < > : + = \ ( ) $ %
  #& ^ * # _ - safe. This removes emojis or foreign characters.
  pattern = r"[^a-zA-Z0-9\s.,!?/'\"<>:+=\\()$@;\[\]|%&^*#_`{}~-]"

  text = re.sub(pattern, " ", text);

  return text.strip()

df_processed = df.copy()
for col in df_processed.columns:
    df_processed[col] = df_processed[col].apply(preprocessing)

print("Processed Data:")

# Set display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# EDA work
display(df_processed)

# create csv
df_processed.to_csv("group1_preprocessed.csv", index=False)